# 04 - Reservation Lifecycle and Resource Occupation

## Objetivo
Observar o ciclo de vida real de uma reserva SeQUeNCe (`Reservation`,
`MemoryTimeCard`, `Rule`) - criação, janela de tempo, memórias alocadas,
caminho, aceitação, ocupação de recursos e liberação ao final - e
demonstrar um caso real de rejeição por insuficiência de recursos.

## Conceitos
- `NetworkManager.request(...)` (chamado por `IntentRequestApp.start`)
  cria um objeto `Reservation` (`sequence/network_management/reservation.py`)
  com `start_time`/`end_time`/`memory_size`/`fidelity`/`path` reais, e
  negocia sua aceitação via `RSVPProtocol` salto a salto.
- Cada memória física tem um `MemoryTimeCard` próprio
  (`sequence/network_management/memory_timecard.py`) que registra quais
  reservas ocupam aquela memória em qual janela de tempo -
  `RSVPProtocol.schedule` rejeita a reserva inteira se não houver
  timecards livres suficientes num nó do caminho.
- Reservas aceitas geram `Rule`s instaladas em
  `ResourceManager.rule_manager` (`sequence/resource_management/rule_manager.py`)
  que efetivamente disparam geração/swap/purificação; uma `Rule` expira
  automaticamente num evento agendado para `reservation.end_time`
  (`ResourceManager.expire`, `sequence/resource_management/resource_manager.py`),
  liberando o nó para outras reservas - **isso não é feito manualmente por
  este notebook**, é o comportamento nativo do simulador.
- Este notebook roda a mesma reserva aceita duas vezes com `stop_time_s`
  diferentes: uma parada **dentro** da janela `[start_time, end_time)`
  (mostra a reserva ativa: regras instaladas, memórias ocupadas) e uma
  parada **depois** de `end_time` (mostra a liberação: regras expiradas).


## Configuração


In [1]:
from sequence.constants import SECOND
from sequence.utils import metrics
from ibqn.demos.topologies import three_node_spec, two_node_spec
from ibqn.network.sequence_adapter import SequenceAdapter
from ibqn.network.capabilities import NetworkCapabilities
from ibqn.execution.sequence_executor import SequenceExecutor
from ibqn.intent.repository import IntentRepository
from ibqn.intent.models import (
    EntanglementIntent, IntentEndpoints, IntentRequirements, IntentValidation, SuccessCondition,
)
from ibqn.planning.planner import IntentPlanner
from ibqn.assurance.telemetry import collect_intent_evidence
from ibqn.demos.tables import reservations_table

SEED = 0
REQUESTED_PAIRS = 10
MIN_FIDELITY = 0.65
START_TIME_S = 0.02
DURATION_S = 0.1  # reservation window: [0.02, 0.12) s

# same topology/physics, only stop_time_s differs between the two runs below
spec_mid = three_node_spec(end_memories=REQUESTED_PAIRS, repeater_memories=REQUESTED_PAIRS * 2, stop_time_s=0.06)
spec_full = three_node_spec(end_memories=REQUESTED_PAIRS, repeater_memories=REQUESTED_PAIRS * 2, stop_time_s=0.2)

intent = EntanglementIntent(
    id="intent-reservation-demo",
    endpoints=IntentEndpoints(source="a", destination="b"),
    requirements=IntentRequirements(
        min_fidelity=MIN_FIDELITY, min_throughput=1, max_latency=1.0,
        requested_pairs=REQUESTED_PAIRS, start_time=START_TIME_S, duration=DURATION_S,
    ),
    validation=IntentValidation(
        metrics=["delivered_pairs"],
        success_conditions=[SuccessCondition(metric="delivered_pairs", operator=">=", expected=1)],
    ),
)

capabilities = NetworkCapabilities(spec_mid)  # stop_time_s does not affect routing/feasibility estimates
planner = IntentPlanner(capabilities)
plan = planner.plan(intent)
print(f"route: {' -> '.join(plan.route)}")
print(f"reservation window: [{START_TIME_S}, {START_TIME_S + DURATION_S}) s")
reservations_table(plan)


2026-10-02 15:03:20,051 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'requested_pairs' -> 'reserved_memory_slots' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:20,058 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'start_time' -> 'start_time_s' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:20,059 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'duration' -> 'duration_s' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:20,065 INFO     ibqn.ibqn.planning.planner intent_id=intent-reservation-demo - planned route a->r->b, estimated_fidelity=0.7300, purification=False


route: a -> r -> b
reservation window: [0.02, 0.12000000000000001) s


,node_id,memories_required,memories_available,satisfied
0,a,10,10,True
1,r,20,20,True
2,b,10,10,True


## Execução

`run_and_inspect` builds a fresh `Timeline`/`SequenceAdapter` for `spec`,
deploys the same `intent`/`plan`, runs to `spec.stop_time_s`, and then
inspects the **real** SeQUeNCe reservation objects at the source router.


In [2]:
def run_and_inspect(spec, label):
    metrics.configure()  # fresh storage: isolates this run's diagnostics from the other one
    adapter = SequenceAdapter(spec, seed=SEED)
    repository = IntentRepository()
    executor = SequenceExecutor(adapter, repository)
    executor.deploy(intent, plan)
    executor.run()

    source_router = adapter.get_router(intent.endpoints.source)
    rsvp = source_router.network_manager.protocol_stack[-1]
    reservation = rsvp.accepted_reservations[0] if rsvp.accepted_reservations else None
    installed_rules = list(source_router.resource_manager.rule_manager.rules)
    reserved_timecards = (
        sum(1 for card in source_router.network_manager.timecards if reservation in card.reservations)
        if reservation is not None else 0
    )
    evidence = collect_intent_evidence(intent)

    return {
        "label": label,
        "stop_time_s": spec.stop_time_s,
        "status": repository.get(intent.id).lifecycle.status,
        "reservation": reservation,
        "installed_rules_count": len(installed_rules),
        "reserved_timecards_at_source": reserved_timecards,
        "delivered_pairs": len(evidence.delivered_pairs),
    }

mid_result = run_and_inspect(spec_mid, "mid-window (stops inside [start_time, end_time))")
full_result = run_and_inspect(spec_full, "post-window (stops after end_time)")


2026-10-02 15:03:20,122 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - topology built: 3 routers, 2 quantum links, formalism=bell_diagonal, seed=0


2026-10-02 15:03:20,125 INFO     ibqn.ibqn.intent.repository intent_id=intent-reservation-demo - intent registered


2026-10-02 15:03:20,126 INFO     ibqn.ibqn.intent.repository intent_id=intent-reservation-demo - intent transitioned to VALIDATED (schema validated)


2026-10-02 15:03:20,127 INFO     ibqn.ibqn.intent.repository intent_id=intent-reservation-demo - intent transitioned to PLANNING (invoking IntentPlanner)


2026-10-02 15:03:20,129 INFO     ibqn.ibqn.intent.repository intent_id=intent-reservation-demo - intent transitioned to PLANNED (selected by ShortestHopCountRouting; hop fidelities [0.85, 0.85])


...

2026-10-02 15:03:22,504 INFO     ibqn.ibqn.intent.repository intent_id=intent-reservation-demo - intent transitioned to PLANNED (selected by ShortestHopCountRouting; hop fidelities [0.85, 0.85])


2026-10-02 15:03:22,506 INFO     ibqn.ibqn.intent.repository intent_id=intent-reservation-demo - intent transitioned to DEPLOYING (applying route a->r->b and submitting reservation to NetworkManager (purification_mode=until_target))


2026-10-02 15:03:22,508 INFO     ibqn.ibqn.execution.sequence_executor intent_id=intent-reservation-demo - submitting reservation a -> b, reserved_memory_slots=10, min_fidelity=0.650, purification_mode=until_target


2026-10-02 15:03:22,509 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - running simulation, stop_time=0.200s


2026-10-02 15:03:22,512 INFO     ibqn.ibqn.intent.repository intent_id=intent-reservation-demo - intent transitioned to ACTIVE (reservation approved)


## Resultados


In [3]:
reservation = mid_result["reservation"]
assert reservation is not None, "reservation must have been accepted"
print("Accepted Reservation (real SeQUeNCe object, both runs produce an equal one):")
print(f"  initiator={reservation.initiator}, responder={reservation.responder}")
print(f"  path={reservation.path}")
print(f"  start_time={reservation.start_time / SECOND:.4f} s, end_time={reservation.end_time / SECOND:.4f} s")
print(f"  memory_size={reservation.memory_size} pairs, target_fidelity={reservation.fidelity}")


Accepted Reservation (real SeQUeNCe object, both runs produce an equal one):
  initiator=a, responder=b
  path=['a', 'r', 'b']
  start_time=0.0200 s, end_time=0.1200 s
  memory_size=10 pairs, target_fidelity=0.65


In [4]:
import pandas as pd

lifecycle_df = pd.DataFrame([
    {
        "run": r["label"],
        "stop_time_s": r["stop_time_s"],
        "final_status": r["status"],
        "installed_rules (source node)": r["installed_rules_count"],
        "reserved_timecards (source node)": r["reserved_timecards_at_source"],
        "delivered_pairs": r["delivered_pairs"],
    }
    for r in (mid_result, full_result)
])
lifecycle_df


,run,stop_time_s,final_status,installed_rules (source node),reserved_timecards (source node),delivered_pairs
0,"mid-window (stops inside [start_time, end_time))",0.06,IntentStatus.ACTIVE,3,10,254
1,post-window (stops after end_time),0.20,IntentStatus.ACTIVE,0,10,665


In [5]:
assert mid_result["installed_rules_count"] > 0, (
    "mid-window run must still have installed rules: the reservation's end_time has not been reached yet"
)
assert full_result["installed_rules_count"] == 0, (
    "post-window run must have zero installed rules: SeQUeNCe automatically expires them at reservation.end_time"
)
assert full_result["delivered_pairs"] >= mid_result["delivered_pairs"] > 0
print("Resource occupation while active and automatic release at end_time both confirmed.")


Resource occupation while active and automatic release at end_time both confirmed.


### Caso de rejeição

Um nó com poucas memórias não consegue satisfazer uma reserva que pede
mais pares do que ele tem `MemoryTimeCard`s livres - `RSVPProtocol.schedule`
rejeita a reserva inteira (não parcialmente), e o rejeito se propaga de
volta ao iniciador como uma mensagem `REJECT` real, nunca uma simulação de
falha.


In [6]:
spec_reject = two_node_spec(memories=2, distance_m=1000, attenuation_db_per_m=1e-4, stop_time_s=0.1)

intent_reject = EntanglementIntent(
    id="intent-reservation-rejected",
    endpoints=IntentEndpoints(source="a", destination="b"),
    requirements=IntentRequirements(
        min_fidelity=MIN_FIDELITY, min_throughput=1, max_latency=1.0,
        requested_pairs=10, start_time=0.02, duration=0.05,  # asks for 10 pairs, node 'a' only has 2 memories
    ),
    validation=IntentValidation(
        metrics=["delivered_pairs"],
        success_conditions=[SuccessCondition(metric="delivered_pairs", operator=">=", expected=1)],
    ),
)

metrics.configure()
adapter_reject = SequenceAdapter(spec_reject, seed=SEED)
repository_reject = IntentRepository()
executor_reject = SequenceExecutor(adapter_reject, repository_reject)
executor_reject.submit(intent_reject)  # submit(): no planner pre-check, so SeQUeNCe's own rejection is what we observe
executor_reject.run()

record = repository_reject.get(intent_reject.id)
print(f"final status: {record.lifecycle.status}")
for transition in record.lifecycle.history:
    print(f"  {transition.from_status} -> {transition.to_status}: {transition.reason}")


2026-10-02 15:03:29,707 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'requested_pairs' -> 'reserved_memory_slots' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:29,710 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'start_time' -> 'start_time_s' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:29,711 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'duration' -> 'duration_s' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:29,717 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - topology built: 2 routers, 1 quantum links, formalism=bell_diagonal, seed=0


2026-10-02 15:03:29,720 INFO     ibqn.ibqn.intent.repository intent_id=intent-reservation-rejected - intent registered


...

2026-10-02 15:03:29,725 INFO     ibqn.ibqn.intent.repository intent_id=intent-reservation-rejected - intent transitioned to DEPLOYING (submitting reservation to NetworkManager)


2026-10-02 15:03:29,727 INFO     ibqn.ibqn.execution.sequence_executor intent_id=intent-reservation-rejected - submitting reservation a -> b, reserved_memory_slots=10, min_fidelity=0.650, purification_mode=until_target


2026-10-02 15:03:29,729 INFO     ibqn.ibqn.intent.repository intent_id=intent-reservation-rejected - intent transitioned to FAILED (reservation rejected by NetworkManager)


2026-10-02 15:03:29,731 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - running simulation, stop_time=0.100s


final status: IntentStatus.FAILED
  None -> IntentStatus.RECEIVED: intent received
  IntentStatus.RECEIVED -> IntentStatus.VALIDATED: schema validated
  IntentStatus.VALIDATED -> IntentStatus.PLANNING: no planner used: delegating route/purification decisions to SeQUeNCe's default reservation mechanism
  IntentStatus.PLANNING -> IntentStatus.PLANNED: trivial pass-through plan
  IntentStatus.PLANNED -> IntentStatus.DEPLOYING: submitting reservation to NetworkManager
  IntentStatus.DEPLOYING -> IntentStatus.FAILED: reservation rejected by NetworkManager


In [7]:
from ibqn.intent.models import IntentStatus

assert record.lifecycle.status == IntentStatus.FAILED, "insufficient memory at the source node must reject the reservation"
print("Rejection due to insufficient resources confirmed (real RSVPProtocol.schedule() failure, not simulated).")


Rejection due to insufficient resources confirmed (real RSVPProtocol.schedule() failure, not simulated).


## Interpretação
A mesma reserva aceita, observada em dois instantes de parada diferentes,
mostra o ciclo completo: enquanto `stop_time_s` está dentro de
`[start_time, end_time)`, a `Rule` instalada pelo `ResourceManager`
continua ativa e memórias seguem sendo ocupadas/entregues; assim que a
simulação passa de `end_time`, o evento de expiração automática já
disparou e `rule_manager.rules` fica vazio - a "liberação ao final" não é
um artefato deste notebook, é o próprio SeQUeNCe agendando
`Event(reservation.end_time, resource_manager.expire, [rule])` no momento
em que a regra é instalada. O caso de rejeição mostra que
`RSVPProtocol.schedule` recusa a reserva inteira (não parcialmente) quando
um nó do caminho não tem `MemoryTimeCard`s livres suficientes - aqui o nó
`a` (2 memórias) não consegue suprir as 10 pares solicitadas.

## Limitações
- `reserved_timecards_at_source` conta timecards cuja lista de reservas
  contém esta reserva por igualdade de objeto/valor
  (`Reservation.__eq__`), não por índice físico de memória - suficiente
  para contagem, não para rastrear qual memória física específica está
  quais.
- Este notebook não usa `MemoryLifecycleRecorder`
  (`ibqn.demos.instrumentation`) - o estado detalhado de cada memória
  (RAW/OCCUPIED/ENTANGLED/PURIFIED) é o tema do próximo notebook
  (`05_sequence_memory_lifecycle.ipynb`).

## Conclusão
Ciclo de vida de uma reserva demonstrado de ponta a ponta com objetos
reais do SeQUeNCe (`Reservation`, `MemoryTimeCard`, `Rule`), incluindo
ocupação de recursos durante a janela ativa, liberação automática ao
final, e um caso real de rejeição por insuficiência de memória.
